In [2]:
import json
import pickle
import numpy as np
from pathlib import Path
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from collections import Counter

c:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
CHUNKS_PATH = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\unified_semantic_chunks\unified_chunks.json"
)
VECTOR_STORE = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\vector_store"
)
CONFIG_PATH   = VECTOR_STORE / "config.json"
FAISS_PATH    = VECTOR_STORE / "faiss.index"
METADATA_PATH = VECTOR_STORE / "metadata.pkl"

# Retrieval settings
VECTOR_CANDIDATES = 40   # how many FAISS results to pull before reranking
RERANK_CANDIDATES = 25   # how many to pass to cross-encoder
TOP_K_DEFAULT     = 5    # final results returned

# Hybrid score weights
W_VECTOR  = 0.6
W_BM25    = 0.3
# Note: remaining 0.1 is reserved for type-boost headroom

# Final score weights (hybrid vs reranker)
W_HYBRID  = 0.7
W_RERANK  = 0.3

print(f"📂 Chunks path   : {CHUNKS_PATH}")
print(f"📂 Vector store  : {VECTOR_STORE}")

📂 Chunks path   : C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\unified_semantic_chunks\unified_chunks.json
📂 Vector store  : C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\vector_store


In [4]:
# Load the config saved during embedding so retriever uses same model
if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)
    MODEL_NAME = config["model_name"]
    print(f"✅ Config loaded")
    print(f"   Model      : {MODEL_NAME}")
    print(f"   Vectors    : {config['total_vectors']}")
    print(f"   Dimension  : {config['dimension']}")
    print(f"   Normalised : {config['normalised']}")
else:
    # Fallback if config wasn't saved
    MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
    print(f"⚠️  No config found — using default model: {MODEL_NAME}")

✅ Config loaded
   Model      : sentence-transformers/all-MiniLM-L6-v2
   Vectors    : 673
   Dimension  : 384
   Normalised : True


In [5]:
# Load raw chunks (for BM25)
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

chunks = [c for c in all_chunks if c.get("text", "").strip()]
print(f"✅ Loaded {len(chunks)} chunks from disk")

# Load enriched chunks (what was actually embedded — may include boosted duplicates)
with open(METADATA_PATH, "rb") as f:
    enriched_chunks = pickle.load(f)
print(f"✅ Loaded {len(enriched_chunks)} enriched/boosted chunks from vector store")

# Load FAISS index
index = faiss.read_index(str(FAISS_PATH))
print(f"✅ FAISS index loaded — {index.ntotal} vectors, dim={index.d}")

✅ Loaded 627 chunks from disk
✅ Loaded 673 enriched/boosted chunks from vector store
✅ FAISS index loaded — 673 vectors, dim=384


In [6]:
print(f"🔄 Loading embedding model: {MODEL_NAME}")
embedding_model = SentenceTransformer(MODEL_NAME)

print(f"🔄 Loading reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("✅ Models loaded")

🔄 Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
🔄 Loading reranker...
✅ Models loaded


In [7]:
# BM25 is built on raw chunks (not enriched/boosted)
# This avoids keyword inflation from duplicated operational chunks
tokenized_corpus = [c["text"].lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

print(f"✅ BM25 index built on {len(tokenized_corpus)} chunks")

✅ BM25 index built on 627 chunks


In [8]:
def detect_document_type(chunk: dict) -> str:
    """
    Uses chunk_type metadata first (set during chunking — most reliable).
    Falls back to structured_data inspection.
    """
    chunk_type = chunk.get("metadata", {}).get("chunk_type", "")

    if chunk_type in ("schema_overview", "schema_core_columns", "schema_extra_columns"):
        return "TABLE_SCHEMA"

    if chunk_type in ("wms_overview", "wms_join_logic", "wms_procedure", "wms_safety_rules"):
        return "OPERATIONAL_REFERENCE"

    if chunk_type in ("text_prose", "text_table"):
        return "TEXT"

    # Fallback
    structured = chunk.get("structured_data")
    if isinstance(structured, dict):
        if "columns" in structured:
            return "TABLE_SCHEMA"
        if "procedures" in structured or "core_tables" in structured:
            return "OPERATIONAL_REFERENCE"

    return "TEXT"

In [9]:
SCHEMA_KEYWORDS = {
    "sql", "select", "query", "join", "where", "insert", "update",
    "column", "columns", "table", "schema", "foreign key", "primary key",
    "field", "fields", "datatype", "varchar", "integer", "index"
}

OPERATIONAL_KEYWORDS = {
    "reverse", "reset", "grn", "receipt", "shipment", "mission",
    "cancel", "validate", "close", "reopen", "resend", "loading",
    "inbound", "outbound", "picking", "putaway", "stock", "movement"
}

def classify_query(query: str) -> dict:
    """
    Classifies query intent so the retriever can apply appropriate boosts.
    Returns dict with flags and matched keywords for transparency.
    """
    query_lower = query.lower()

    schema_hits      = [k for k in SCHEMA_KEYWORDS      if k in query_lower]
    operational_hits = [k for k in OPERATIONAL_KEYWORDS if k in query_lower]

    is_schema      = len(schema_hits) > 0
    is_operational = len(operational_hits) > 0

    # If both match, prefer operational (more specific in your WMS context)
    if is_schema and is_operational:
        is_schema = False

    return {
        "is_schema"        : is_schema,
        "is_operational"   : is_operational,
        "schema_hits"      : schema_hits,
        "operational_hits" : operational_hits
    }

In [10]:
def retrieve_context(query: str, top_k: int = TOP_K_DEFAULT, verbose: bool = False) -> tuple:
    """
    Hybrid retrieval pipeline:
      1. FAISS vector search (semantic)
      2. BM25 keyword search
      3. Hybrid score with intent-based boosts
      4. Cross-encoder reranking
    
    Key fix: FAISS searches enriched_chunks (what was embedded),
    BM25 searches raw chunks (no duplicates).
    Scores are merged carefully to avoid index mismatch.
    """
    query_lower  = query.lower()
    query_tokens = query_lower.split()
    intent       = classify_query(query)

    if verbose:
        print(f"🔍 Intent: schema={intent['is_schema']}, "
              f"operational={intent['is_operational']}")
        print(f"   Schema hits      : {intent['schema_hits']}")
        print(f"   Operational hits : {intent['operational_hits']}")

    # --- Step 1: Vector search (over enriched/boosted index) ---
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, VECTOR_CANDIDATES)

    # --- Step 2: BM25 over raw chunks ---
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_max    = bm25_scores.max() if bm25_scores.max() > 0 else 1.0
    bm25_norm   = bm25_scores / bm25_max  # normalise to [0, 1]

    # --- Step 3: Merge scores ---
    seen_texts = {}  # deduplicate by text content (handles boosted duplicates)

    for vector_score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        # enriched_chunks aligns with FAISS index
        chunk    = enriched_chunks[idx]
        metadata = chunk.get("metadata", {})
        doc_type = detect_document_type(chunk)
        text     = chunk["text"]

        # Skip if we've already seen this text (from boosted duplicates)
        if text in seen_texts:
            continue

        # Map back to raw chunk index for BM25 score
        # Match by source + chunk_id (reliable since chunker assigns these)
        source   = metadata.get("source", "")
        chunk_id = chunk.get("chunk_id", -1)

        # Find corresponding raw chunk BM25 score
        raw_idx = next(
            (i for i, c in enumerate(chunks)
             if c.get("metadata", {}).get("source") == source
             and c.get("chunk_id") == chunk_id),
            None
        )
        bm25_score = float(bm25_norm[raw_idx]) if raw_idx is not None else 0.0

        # Normalise vector score from [-1,1] to [0,1]
        v_score_norm = (float(vector_score) + 1) / 2

        hybrid = (W_VECTOR * v_score_norm) + (W_BM25 * bm25_score)

        # --- Intent-based boosts ---
        if intent["is_operational"] and doc_type == "OPERATIONAL_REFERENCE":
            hybrid += 0.4
        if intent["is_schema"] and doc_type == "TABLE_SCHEMA":
            hybrid += 0.3

        # Table name match boost (direct mention in query)
        table_name = metadata.get("table_name", "")
        if table_name and table_name.lower() in query_lower:
            hybrid += 0.5

        # Procedure name match boost
        proc_name = metadata.get("procedure_name", "")
        if proc_name and proc_name.lower() in query_lower:
            hybrid += 0.4

        # Related table mention boost
        related_tables = metadata.get("related_tables", [])
        for rt in related_tables:
            if rt and rt.lower() in query_lower:
                hybrid += 0.2
                break

        seen_texts[text] = {
            "hybrid_score"   : hybrid,
            "vector_score"   : v_score_norm,
            "bm25_score"     : bm25_score,
            "doc_type"       : doc_type,
            "text"           : text,
            "metadata"       : metadata,
            "structured_data": chunk.get("structured_data")
        }

    if not seen_texts:
        return [], 0.0

    # Sort by hybrid score, take top candidates for reranking
    results    = sorted(seen_texts.values(), key=lambda x: x["hybrid_score"], reverse=True)
    candidates = results[:RERANK_CANDIDATES]

    # --- Step 4: Cross-encoder reranking ---
    pairs         = [(query, r["text"]) for r in candidates]
    rerank_scores = reranker.predict(pairs)

    for i, r in enumerate(candidates):
        r["rerank_score"] = float(rerank_scores[i])
        r["final_score"]  = (W_HYBRID * r["hybrid_score"]) + (W_RERANK * r["rerank_score"])

    candidates = sorted(candidates, key=lambda x: x["final_score"], reverse=True)
    top_results = candidates[:top_k]

    confidence = float(np.mean([r["final_score"] for r in top_results]))

    return top_results, confidence

In [11]:
def print_results(query: str, top_k: int = TOP_K_DEFAULT, verbose: bool = False):
    print(f"\n{'='*80}")
    print(f"🔎 QUERY: {query}")
    print(f"{'='*80}")

    results, confidence = retrieve_context(query, top_k=top_k, verbose=verbose)

    if not results:
        print("❌ No results found")
        return

    print(f"📊 Confidence: {confidence:.4f}  |  Results: {len(results)}")

    for i, r in enumerate(results, 1):
        meta = r["metadata"]
        print(f"\n  Rank {i}")
        print(f"  {'─'*70}")
        print(f"  doc_type     : {r['doc_type']}")
        print(f"  chunk_type   : {meta.get('chunk_type', 'unknown')}")
        print(f"  final_score  : {r['final_score']:.4f}")
        print(f"  hybrid_score : {r['hybrid_score']:.4f}")
        print(f"  rerank_score : {r['rerank_score']:.4f}")
        print(f"  source       : {meta.get('source', 'unknown')}")
        print(f"  category     : {meta.get('category', 'unknown')}")
        if meta.get("table_name"):
            print(f"  table        : {meta.get('table_name')}")
        if meta.get("procedure_name"):
            print(f"  procedure    : {meta.get('procedure_name')}")
        if meta.get("page_number"):
            print(f"  page         : {meta.get('page_number')}")
        print(f"  text preview :")
        print(f"    {r['text'][:200].strip()}...")

In [12]:
test_queries = [
    # Operational
    "How do I reverse a GRN?",
    "Reset a mission in Speed",
    "Resend outbound shipment",
    # Schema
    "What is the primary key of REE_DAT?",
    "Show join between REE_DAT and DOS_DAT",
    "List columns of REE_DAT",
    "How does receipt relate to STK_DAT table?",
    # Text/General
    "Explain warehouse picking process"
]

for query in test_queries:
    print_results(query, top_k=3, verbose=False)


🔎 QUERY: How do I reverse a GRN?


📊 Confidence: 2.3911  |  Results: 3

  Rank 1
  ──────────────────────────────────────────────────────────────────────
  doc_type     : OPERATIONAL_REFERENCE
  chunk_type   : wms_safety_rules
  final_score  : 2.8835
  hybrid_score : 1.0807
  rerank_score : 7.0902
  source       : Reverse Closed GRN.json
  category     : Speed Support Document
  text preview :
    DOCUMENT: Reverse Closed GRN — SAFETY RULES

  1. Always execute SELECT before UPDATE.
  2. Do not execute UPDATE in production without approval.
  3. Support-level SQL only.
  4. Follow inbound/outbo...

  Rank 2
  ──────────────────────────────────────────────────────────────────────
  doc_type     : OPERATIONAL_REFERENCE
  chunk_type   : wms_procedure
  final_score  : 2.1466
  hybrid_score : 1.0399
  rerank_score : 4.7289
  source       : Reverse Closed GRN.json
  category     : Speed Support Document
  procedure    : Reverse Closed GRN
  text preview :
    PROCEDURE: Reverse Closed GRN
DOCUMENT: Reverse Closed GRN
CATEGORY

In [ ]:
# Run this cell on its own whenever you want to test a custom query
query = "how do I do picking in speed using RF?"
print_results(query, top_k=5, verbose=True)


🔎 QUERY: how do I do picking in speed using RF ?
🔍 Intent: schema=False, operational=True
   Schema hits      : []
   Operational hits : ['picking']
📊 Confidence: 0.2011  |  Results: 5

  Rank 1
  ──────────────────────────────────────────────────────────────────────
  doc_type     : TEXT
  chunk_type   : text_prose
  final_score  : 0.7887
  hybrid_score : 0.7157
  rerank_score : 0.9589
  source       : Speed WMS - SOP Knowledge Base (RAG-Ready Structure).txt
  category     : Speed Support Document
  page         : 13
  text preview :
    1. Scan location.
2. Scan supports.
3. Enter counted quantities.
Resolve Inventory Discrepancies
System compares:
Recorded stock
vs
Counted stock
Discrepancies generate adjustment movements.
Validate...

  Rank 2
  ──────────────────────────────────────────────────────────────────────
  doc_type     : TEXT
  chunk_type   : text_prose
  final_score  : 0.5829
  hybrid_score : 0.5484
  rerank_score : 0.6635
  source       : LOREAL_MAROC_MACASLEA1.txt
  